# Notebook 06: Final Load & KPI Preparation for Tableau
**Purpose:** Compute all KPIs, create summary tables, and 
export perfectly formatted CSV files that Tableau can 
load without any issues.
**Output files:** All saved to data/processed/

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/student_health_features.csv')
print(f"Loaded cleaned data: {df.shape[0]:,} rows × {df.shape[1]} cols")

os.makedirs('../data/processed', exist_ok=True)

Loaded cleaned data: 1,000,000 rows × 25 cols


In [2]:
print("Computing KPIs...")

kpis = {}

kpis['total_students'] = len(df)

kpis['avg_burnout_composite'] = round(df['burnout_composite'].mean(), 2)

kpis['pct_high_stress'] = round(
    (df['stress_level'] >= 7).sum() / len(df) * 100, 1
)

kpis['avg_sleep_hours'] = round(df['sleep_hours'].mean(), 2)

kpis['pct_sleep_deprived'] = round(
    (df['sleep_hours'] < 7).sum() / len(df) * 100, 1
)

kpis['avg_financial_stress'] = round(df['financial_stress'].mean(), 2)

kpis['pct_high_financial_stress'] = round(
    (df['financial_stress'] >= 7).sum() / len(df) * 100, 1
)

kpis['avg_wellbeing_index'] = round(df['wellbeing_index'].mean(), 2)

kpis['pct_vulnerability_critical'] = round((df['vulnerability_score'] >= 4).sum() / len(df) * 100, 1)

kpis['pct_burnout_risk'] = round(
    (df['burnout_composite'] >= 6.5).sum() / len(df) * 100, 1
)

print("\n" + "=" * 50)
print("  PROJECT KPI DASHBOARD")
print("=" * 50)
for kpi_name, kpi_value in kpis.items():
    label = kpi_name.replace('_', ' ').title()
    if 'pct' in kpi_name:
        print(f"  {label:<35} {kpi_value}%")
    elif 'total' in kpi_name:
        print(f"  {label:<35} {kpi_value:,}")
    else:
        print(f"  {label:<35} {kpi_value}")
print("=" * 50)

kpi_df = pd.DataFrame([kpis]).T.reset_index()
kpi_df.columns = ['KPI_Name', 'KPI_Value']
kpi_df.to_csv('../data/processed/kpi_summary.csv', index=False)
print("\nKPI summary saved to data/processed/kpi_summary.csv")

Computing KPIs...

  PROJECT KPI DASHBOARD
  Total Students                      1,000,000
  Avg Burnout Composite               2.84
  Pct High Stress                     5.1%
  Avg Sleep Hours                     6.5
  Pct Sleep Deprived                  63.0%
  Avg Financial Stress                5.0
  Pct High Financial Stress           15.9%
  Avg Wellbeing Index                 5.17
  Pct Vulnerability Critical          2.8%
  Pct Burnout Risk                    0.3%

KPI summary saved to data/processed/kpi_summary.csv


In [3]:
print("Exporting main Tableau dataset...")
df.to_csv('../data/processed/student_health_tableau_ready.csv', 
          index=False)
print(f"Main file: {len(df):,} rows × {df.shape[1]} columns")

burnout_by_year = df.groupby('academic_year').agg(
    avg_burnout        = ('burnout_composite', 'mean'),
    avg_stress         = ('stress_level', 'mean'),
    
    
    student_count      = ('burnout_composite', 'count'),
    pct_high_stress    = ('stress_level', lambda x: (x>=7).mean()*100)
).round(3).reset_index()

burnout_by_year.to_csv(
    '../data/processed/kpi_burnout_by_year.csv', index=False
)
print(f"Burnout by year: {burnout_by_year.shape}")
print(burnout_by_year.to_string(index=False))

stress_dist = df['stress_tier'].value_counts(normalize=True) \
                .reset_index()
stress_dist.columns = ['stress_tier', 'proportion']
stress_dist['percentage'] = (stress_dist['proportion'] * 100).round(1)
stress_dist['student_count'] = (
    stress_dist['proportion'] * len(df)
).astype(int)
stress_dist.to_csv(
    '../data/processed/kpi_stress_tier_distribution.csv', index=False
)
print(f"\nStress tier distribution:")
print(stress_dist.to_string(index=False))

df['sleep_rounded'] = df['sleep_hours'].round(0)
sleep_anxiety = df.groupby('sleep_rounded').agg(
    avg_wellbeing = ('wellbeing_index', 'mean'),
    avg_stress     = ('stress_level', 'mean'),
    avg_burnout    = ('burnout_composite', 'mean'),
    student_count  = ('wellbeing_index', 'count')
).round(3).reset_index()
sleep_anxiety.to_csv(
    '../data/processed/kpi_sleep_vs_anxiety.csv', index=False
)
print(f"\nSleep vs anxiety summary: {sleep_anxiety.shape}")

df['financial_rounded'] = df['financial_stress'].round(0)
fin_dep = df.groupby('financial_rounded').agg(
    avg_academic_pressure = ('academic_pressure_index', 'mean'),
    avg_burnout    = ('burnout_composite', 'mean'),
    student_count  = ('academic_pressure_index', 'count')
).round(3).reset_index()
fin_dep.to_csv(
    '../data/processed/kpi_financial_vs_depression.csv', index=False
)
print(f"Financial vs depression summary: {fin_dep.shape}")

gender_summary = df.groupby('gender').agg(
    avg_burnout    = ('burnout_composite', 'mean'),
    avg_wellbeing = ('wellbeing_index', 'mean'),
    avg_academic_pressure = ('academic_pressure_index', 'mean'),
    avg_stress     = ('stress_level', 'mean'),
    avg_sleep      = ('sleep_hours', 'mean'),
    student_count  = ('burnout_composite', 'count')
).round(3).reset_index()
gender_summary.to_csv(
    '../data/processed/kpi_gender_summary.csv', index=False
)
print(f"Gender summary: {gender_summary.shape}")
print(gender_summary.to_string(index=False))

heatmap_data = df.pivot_table(
    index='stress_tier',
    columns='academic_year',
    values='stress_level',
    aggfunc='count'
).fillna(0)

heatmap_pct = heatmap_data.div(heatmap_data.sum(axis=0), axis=1) * 100
heatmap_pct = heatmap_pct.round(1).reset_index()
heatmap_pct.to_csv(
    '../data/processed/kpi_heatmap_stress_by_year.csv', index=False
)
print(f"Heatmap data exported")

Exporting main Tableau dataset...


Main file: 1,000,000 rows × 25 columns
Burnout by year: (4, 5)
academic_year  avg_burnout  avg_stress  student_count  pct_high_stress
       Year 1        2.833       4.243         249671            5.074
       Year 2        2.836       4.245         249886            5.180
       Year 3        2.838       4.250         250433            5.107
       Year 4        2.837       4.247         250010            5.094

Stress tier distribution:
 stress_tier  proportion  percentage  student_count
Medium (4–6)    0.619242        61.9         619242
   Low (0–3)    0.231184        23.1         231184
 High (7–10)    0.149574        15.0         149574

Sleep vs anxiety summary: (8, 5)
Financial vs depression summary: (11, 4)
Gender summary: (3, 7)
gender  avg_burnout  avg_wellbeing  avg_academic_pressure  avg_stress  avg_sleep  student_count
Female        2.836          5.167                  5.631       4.246      6.503         480070
  Male        2.836          5.166                  5.628

Heatmap data exported


In [4]:
print("\n" + "=" * 60)
print("  ALL EXPORTS COMPLETE")
print("=" * 60)

export_files = [
    'student_health_features.csv',
    'student_health_tableau_ready.csv',
    'kpi_summary.csv',
    'kpi_burnout_by_year.csv',
    'kpi_stress_tier_distribution.csv',
    'kpi_sleep_vs_anxiety.csv',
    'kpi_financial_vs_depression.csv',
    'kpi_gender_summary.csv',
    'kpi_heatmap_stress_by_year.csv',
]

for filename in export_files:
    filepath = f'../data/processed/{filename}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / 1024
        print(f"  {filename:<45} ({size:.1f} KB)")
    else:
        print(f"  MISSING: {filename}")

print("\n" + "=" * 60)
print("  PROJECT COMPLETE — READY FOR TABLEAU")
print("=" * 60)
print("""
FINAL CHECKLIST BEFORE TABLEAU:
  data/raw/student_health_raw.csv      — never edited
  data/processed/student_health_features.csv — your ETL output
  data/processed/student_health_tableau_ready.csv — for Tableau
  data/processed/kpi_*.csv            — all KPI tables
  All 5 notebooks committed to GitHub
  Every transformation logged in 02_cleaning.ipynb

NEXT STEP: Open Tableau Public and connect to:
  data/processed/student_health_tableau_ready.csv
""")


  ALL EXPORTS COMPLETE
  student_health_features.csv                   (241032.0 KB)
  student_health_tableau_ready.csv              (241032.0 KB)
  kpi_summary.csv                               (0.3 KB)
  kpi_burnout_by_year.csv                       (0.2 KB)
  kpi_stress_tier_distribution.csv              (0.1 KB)
  kpi_sleep_vs_anxiety.csv                      (0.3 KB)
  kpi_financial_vs_depression.csv               (0.3 KB)
  kpi_gender_summary.csv                        (0.2 KB)
  kpi_heatmap_stress_by_year.csv                (0.1 KB)

  PROJECT COMPLETE — READY FOR TABLEAU

FINAL CHECKLIST BEFORE TABLEAU:
  data/raw/student_health_raw.csv      — never edited
  data/processed/student_health_features.csv — your ETL output
  data/processed/student_health_tableau_ready.csv — for Tableau
  data/processed/kpi_*.csv            — all KPI tables
  All 5 notebooks committed to GitHub
  Every transformation logged in 02_cleaning.ipynb

NEXT STEP: Open Tableau Public and connect to:
  data/